# Laboratory Exercise 5: Experimental Characterization of TRIK Motors

### Motors M3 and M4 — encoder-based analysis

This notebook examines how two TRIK robot motors respond to different PWM commands. The measurements are stored directly in the notebook, so the analysis can be run without downloading separate data files.

**Measurement setup.** Each motor was tested at four positive command levels: **30%, 50%, 70%, and 100% PWM**. Every run contains 100 encoder-position samples, giving eight recordings in total. A 20 ms interval is used to construct the time axis, with the first reading at $t=0$. This is an assumed sampling interval rather than a hardware-recorded timestamp. Encoder readings are retained in their original units.

**Aim.** Compare the motors, estimate a simple dynamic model for each operating point, and identify what can and cannot be concluded from these measurements.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import base64
import json
import zlib

DT = 0.020  # assumed interval between successive encoder readings
PWMS = [30, 50, 70, 100]
MOTORS = ['M3', 'M4']
# Losslessly encoded motor-position recordings (4 inputs per motor):
EMBEDDED_DATA = "eJxll1uO5TYMRLfSyHd/6P3IHrKCYJCFBNl7dA5l3w4CTN+xZVsSWaxi6e/f/qh/1fTb719/pu+v869+f43vr5zP3/m/lPO3z+j8/mr9+6u38/iMzfN8nf/3+cuJn3ye5MJP5fu2mODMl+eZMa/zbd5nkpI2szJtPWOlMdYZY8rCe2Wfb2s+r9Rybmtl/c7tGOyEW9Zt6dy2ctZolVv215iqsWRbZ+ae2HM+Y71Wdn++6Oyq80pf53akczsyYZfzxWjcGuY42xgEOfZ5b7LxycYnAc5+3pvjvDfn+WyyoZUmWSE19TxY5GCRrTXPNtY6E2w2tNnQZkOb8Dev7Hkm3ZuEpUQWU578VoBIhJYTseVkPhPR5fMSv5mnGZgOCmLRHRmOLN7PhJALSc2leN3Sr+8v0O8P+sQGfucqX9jJHAmpREwM7szJeZPturfB2OTpbnzObCxT2M/Flwf7vFdJU6VUKk9rt77AnE024m5VLIFxgi/rdj7rxSKsL4w82B0EAYoNDXI6QOaCJ25AxmqT1S5kYMRni88uWoMgG0DlFygg62J03ttmPiURKZZ/E7Vubqbj25yThYOLSLVAxPHlOFV+EGG8UAu5kOlchuMr8sv8lV3kWsVGZKs1UElmbon3m5g2EpPbmBKQN3tiRAbkLjE7tZ1NXO5WxZC6w0obbd+qmE9VWG4AC3JAQ9phwcmUBRkRuImX8zwgwJL5qespAvFHXCrLBavJqrE0wAguB/RnX23LYHjL9jtLX9Q3qO+HslTgILMTQk8qOAC3diHBgleryExqmRJZZGkjJJsMXj4uYDY2E3cBtsbTkmCmNeTuZF1oBdikFyr10Ewgm2QTkmLpVAGr0rUKeUX6HjilaLMgWvdasNsWNqHqVTjdTzfVilgeJSAUBkttUMp5El+eFuJUmKfSMI1xie4S/uWKKyTb/WwLdKNnefehio9bIocFt0bAF+5SF3AkxMbN3w5QkpKAvHfEBQZWOFbZVgXvCnq3Bs6EDXyC+WSy81nns5EUa5BnQ7J8ktTJLoPg0MfAFlUTIsxUmwTu5h4VXAmcpa55S9Ivme0IIWLI0/5mnq+AmqsikaoksKYPpqK8gpzzB5pbWo4PjtZVn0UqimPpQUJ+lZKp2k5raUrd6cz2moNaNFdxjB5r/WzFYlshClZJrFis5GKMJYVSK/XGWKRtKcm+XwMvr5ftOkllu3Zl5kNrRlq2jzdGlJ5irZ70MIMtt/SZrZn2H6txZu1XXNrtOdQCsJfoOwTGlDRHmdofwQndVCRD7cyhCbOmCZxdJ7tZf9pRKBFrG4GVuWxMWIuk8UDoiP52J/oU5K6WLJRpSbFCosCrUWKNSdtQtvp1Hw1MO2rX0cde7V3WMg9gfUeYooEBaOiZpU0+ByUWrWxa6euRN1C+ZoSaR8+mqkz4U0bY3sjVytqSje7BAyIK8SOiRZquLUH8eG+zeLQ83tu8t6P3S4CUw/bZGqqYSJLHoYRQ1gv56y/yB2/L5GLdaOYX5/1YSjFWIuVRf+G14NeLLBPRMcJIBqh6SABklosnV0AUUK70ug2gVLuIuXVVjCt42gG6s0Znjc4afdS3A9UHtqL5mC9iOkee0gMnlSKDZ6uvA7ErkX/iCIgg84L7izUWayzW2ElggIP63TiNzQQPJvXTqhTqpICk4bVcSEpyFresmDxO8SNu2XZQop8rfaWG0Hk9vNa7FHlYRVKPnqtSVvv4tDObnVQ5TW3eWvifq5Bx92xBXig3N2zDDJujSNq7HlaHv1wv7P3SWMNUS35PDfvhbYDdXn+RXmspR0GXyr8Qg/2SlO1BN1xlf/tNfk1GejEFYhBf+JXoPPi1FV4J/Ci3jbjsnl6nEYZyfCh1O1HQSLMo7YIQua4fwAVPhCxOYNdKCpla+LiO+QOsOKaZXf1Mc/7m/DbfLBOyFux4jzCR9in52O19Ktihpa4jhwOxc8XhT88zVnQxf/VLkiBry7JKdXpZuuXxcRRXIHQU+Qp/OAq9jPpub6ISWr9Hx2st0XJafmXsp6OoL+AgbJcZ7wnC4x9GZCBDbnlWweXwAA4hp2Rgkf7lCZimEYc5imDruJIeL4nFFUXH75FAWuUhAVcUutiNwEsSZdGx+NWs1x9Kq2v0xSucocmxfF9HIQpWlLJ0HIUuQrmYIjUdt3CzwWXPQsdRDGVWL2EsnoJeRzF0FNFfdRdDd4HolqwmKy8l6zTiTBhQFXZ+erFOw/HqeNUneAw8J9PqydHrHuIefVtfoTPpffz651+L4RJQ"
data = json.loads(zlib.decompress(base64.b64decode(EMBEDDED_DATA)).decode())
measurements = {(m,p): np.asarray(data[f'{m}_{p}'], dtype=float) for m in MOTORS for p in PWMS}
assert all(len(y) == 100 for y in measurements.values())
print('Loaded', len(measurements), 'embedded experiments; total samples:', sum(map(len,measurements.values())))

## A. Checking the encoder recordings

Before fitting any model, I summarize the eight runs to verify their lengths, starting readings, ending readings, and the direction of encoder movement. The monotonicity check reports whether successive position readings are nondecreasing.

In [ ]:
overview = pd.DataFrame([
    {'motor':m,'PWM (%)':p,'samples':len(y),'start':y[0],'final':y[-1],
     'all_increments_nonnegative':bool(np.all(np.diff(y)>=0))}
    for (m,p),y in measurements.items()])
display(overview)

## B. Estimating motor dynamics

For a motor starting from rest, I use a first-order approximation of the angular-speed response:

$$\omega(t)=\omega_{\infty}\left(1-e^{-t/\tau}\right).$$

The corresponding accumulated encoder position can be described by integrating the speed:

$$\theta(t)=\theta_{\mathrm{offset}}+\omega_{\infty}\left[t-\tau\left(1-e^{-t/\tau}\right)\right].$$

Here, $\omega_{\infty}$ represents the estimated steady-state speed, $\tau$ the response time constant, and $\theta_{\mathrm{offset}}$ an initial-position correction. Instead of fitting a model to numerical derivatives, I fit this equation directly to the recorded positions. Robust nonlinear least squares reduces the influence of individual irregular samples. All time-related estimates rely on the nominal 20 ms interval.

In [ ]:
def position_model(t, omega_ss, tau, offset):
    return offset + omega_ss*(t - tau*(-np.expm1(-t/tau)))

fits = {}
rows = []
for (motor, pwm), theta in measurements.items():
    t = np.arange(len(theta))*DT
    tail_speed = np.median(np.diff(theta)[-40:])/DT
    result = least_squares(
        lambda params: position_model(t,*params)-theta,
        x0=[tail_speed,0.15,0],
        bounds=([0,0.005,-80],[10000,2,80]),
        loss='soft_l1',f_scale=5
    )
    omega, tau, offset = result.x
    predicted = position_model(t,*result.x)
    r2 = 1-np.sum((theta-predicted)**2)/np.sum((theta-theta.mean())**2)
    fits[motor,pwm] = {'t':t,'theta':theta,'predicted':predicted,'omega':omega,'tau':tau}
    rows.append({'motor':motor,'PWM (%)':pwm,'tail speed (units/s)':tail_speed,
                 'fitted speed (units/s)':omega,'T_m (s)':tau,
                 'fit offset (units)':offset,'position R²':r2})
summary = pd.DataFrame(rows)
display(summary.round(4))

## C. Position trajectories and approximate speeds

The first set of graphs compares the encoder-position curves for the different PWM values. I also calculate a sample-to-sample speed estimate:

$$v_i\approx\frac{\theta_i-\theta_{i-1}}{\Delta t},\qquad \Delta t=0.020\ \mathrm{s}.$$

Because encoder values are discrete and the sampling times were not independently recorded, this calculated speed may contain spikes or fluctuations that do not reflect actual motor acceleration.

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,8),sharex='col')
for j,m in enumerate(MOTORS):
    for pwm in PWMS:
        f=fits[m,pwm]
        axes[0,j].plot(f['t'],f['theta'],label=f'{pwm}%')
        axes[1,j].plot(f['t'][1:],np.diff(f['theta'])/DT,alpha=.75,label=f'{pwm}%')
    axes[0,j].set_title(f'{m}: recorded encoder position')
    axes[1,j].set_title(f'{m}: estimated sample speed')
    axes[0,j].set_ylabel('Encoder units')
    axes[1,j].set_ylabel('Encoder units/s')
    axes[1,j].set_xlabel('Nominal time (s)')
    for ax in axes[:,j]: ax.grid(alpha=.25); ax.legend(ncol=2)
fig.tight_layout(); plt.show()

## D. Comparing the model with experimental observations

The next plots overlay the calculated first-order response with each encoder recording. Agreement between the curves indicates that the chosen equation describes these datasets well; it does not demonstrate that the physical motor is exactly a first-order system under every operating condition.

In [ ]:
fig, axes=plt.subplots(2,4,figsize=(16,7),sharex=True)
for i,m in enumerate(MOTORS):
    for j,pwm in enumerate(PWMS):
        f=fits[m,pwm]; ax=axes[i,j]
        ax.plot(f['t'],f['theta'],'.',ms=3,label='Encoder data')
        ax.plot(f['t'],f['predicted'],'-',label='First-order fit')
        ax.set_title(f'{m} at {pwm}% | T={f["tau"]:.3f} s')
        ax.grid(alpha=.25)
        if j==0: ax.set_ylabel('Encoder units')
        if i==1: ax.set_xlabel('Nominal time (s)')
axes[0,0].legend(); fig.tight_layout(); plt.show()

## E. How motor behavior varies with PWM

For each experiment, a late-run speed estimate is obtained from the median of the final 40 position increments. I compare that estimate and the fitted time constant across the four PWM inputs.

The results describe only the tested **positive** PWM commands. No reverse-direction tests or low-command sweep were included.

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(12,4.5))
for m in MOTORS:
    s=summary[summary.motor==m]
    axes[0].plot(s['PWM (%)'],s['tail speed (units/s)'],'o-',label=m)
    axes[1].plot(s['PWM (%)'],s['T_m (s)'],'o-',label=m)
axes[0].set_ylabel('Late-time speed (encoder units/s)')
axes[1].set_ylabel('Fitted time constant (s)')
for ax in axes:
    ax.set_xlabel('PWM (%)'); ax.set_xticks(PWMS); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); plt.show()

## F. Direct comparison of M3 and M4

To make the mismatch easier to interpret, the following table reports the final encoder count of each motor at equal PWM settings. It also computes the absolute difference and the percentage by which M3 exceeds M4.

In [ ]:
wide=overview.pivot(index='PWM (%)',columns='motor',values='final')
wide['M3 minus M4 (encoder units)']=wide['M3']-wide['M4']
wide['M3 excess over M4 (%)']=100*(wide['M3']/wide['M4']-1)
display(wide.round(2))

## G. Interpretation and experimental limitations

**Motor electrical parameters.** The encoder trajectories describe motion, but they are not sufficient for estimating a physical back-EMF coefficient $k_e$. That would additionally require a calibrated speed in rad/s and suitable motor voltage/current information. Since these electrical measurements are unavailable, I do not assign a numerical value to $k_e$.

**Nonlinear effects.** The measured speeds rise as the PWM command increases. However, four positive input values cannot locate the dead zone near zero, test reverse rotation, identify saturation outside the sampled range, or fully characterize a $-100\%$ to $+100\%$ PWM response.

**Effect on robot motion.** If the motors power the left and right wheels, unequal responses to identical PWM commands may cause unintended turning during straight-line motion. The available experiments do not include the wheel geometry, a reliable encoder-to-angle calibration, or measured robot trajectories, so a numerical odometry error cannot be calculated.

**Sample timing.** The 20 ms value comes from the intended loop delay rather than measured timestamps. Execution and file-writing overhead could alter the true interval. Consequently, the speed estimates and fitted time constants must be viewed as approximate. Their variation between PWM settings also highlights the limits of the simplified model.

## H. Summary of findings

1. The dataset contains eight usable recordings: four PWM levels for each of the two motors.
2. M3 finishes ahead of M4 at 30%, 50%, and 70% PWM; at 100% the final readings are close.
3. Both motors achieve larger late-run speeds when the positive PWM input is increased.
4. The encoder-position traces support estimating a first-order response separately at each tested setting.
5. Further experiments with actual timestamps, electrical measurements, negative PWM commands, and robot-motion tracking would be needed for a more complete motor identification.

## I. Preparing results for external analysis (optional)

The original measurements and their fitted position values can be assembled into one table. The final two lines for saving CSV files are intentionally commented out; uncomment them if files are needed for a separate report.

In [ ]:
all_rows=[]
for (m,p),f in fits.items():
    for i,(t,y,pred) in enumerate(zip(f['t'],f['theta'],f['predicted'])):
        all_rows.append({'motor':m,'PWM (%)':p,'sample':i,'nominal_time_s':t,
                         'encoder_units':y,'fitted_encoder_units':pred})
all_data=pd.DataFrame(all_rows)
# Enable these statements to write CSV output:
# all_data.to_csv('Lab5_all_measurements.csv',index=False)
# summary.to_csv('Lab5_motor_parameters.csv',index=False)
print('Embedded samples:',len(all_data),'| fitted parameter rows:',len(summary))